# PSALM Tutorial

This notebook shows how to load a PSALM model bundle and scan sequences for domains.
Update paths to match your environment.

## Setup

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve()
sys.path.insert(0, str(repo_root))

from psalm.psalm_model import PSALM

# Load from HuggingFace https://huggingface.co/ProteinSequenceAnnotation/PSALM-2
psalm = PSALM(model_name="ProteinSequenceAnnotation/PSALM-2",
              use_fa=True)

/n/eddy_lab/Lab/protein_annotation_dl/PSALM-2/psalm2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 298.84it/s]


## Scan from FASTA

In [4]:
psalm.scan(fasta="example_fastas/Q09870.fa",
           verbose=True) #Off by default, shows the domain annotations and alignments for each family
# returns domain dict

################################################################################
# date/time: 2026-01-24 01:08:10
# model: ProteinSequenceAnnotation/PSALM-2
# device: cuda:0
# refine_extended: True
# beam size: 64
# embedding time: 24.19ms
# decoding time: 1.81ms
# top-1 emission filter: 2/24076 families passed
# score filter ≥0.00: 2/2 domains passed
################################################################################
>>> Query: Q09870 (363 aa)
--- Pfam domain hits ---
Score   Model        Description                                              Pfam      Start    Stop
-----------------------------------------------------------------------------------------------------
1.000   RTC          RNA 3'-terminal phosphate cyclase                        PF01137       7     355
1.000   RTC_insert   RNA 3'-terminal phosphate cyclase (RTC), insert domain   PF05189     183     283

Domain annotation for each model (and alignments):

>> RTC   RNA 3'-terminal phosphate cyclase   (PF0113

{'Q09870': [('PF01137',
   7,
   355,
   0.99999947991194,
   6203.798876502999,
   1.1606290900865535,
   34.48702592136725,
   'full (merged)'),
  ('PF05189',
   183,
   283,
   0.999991992366746,
   2418.071356489377,
   1.0740433931350708,
   21.45829852637878,
   'full')]}

## Scan from a raw sequence

In [6]:
psalm.scan(sequence="TGGKSYTCTECGKGFIKKSRLVTHMKIHTGETHFICTECGKGFSQKGILQTHMKTHTGEKPFTCTECGKNFAQITTLLRHLTIHTGEKPFSCTECGKHFAHKGHLVSHMKTHTGEKPFTCTECGKHFAQKGHLVSHMKTHTGEKPFTCTECGKNFAQKTNLLCHLKIHTGEKPFTCTECGDKFAKKNNLLRHLKIHTGEKPFTCTECGKAFTLKGSLVGHMKIHTGEKPFSCTQCGKNFTQKNSLLCHLTMHTGEKPFTCTECGKGFALKGNLVLHTKIHTGEKPFSCTQCGKNFAQKNSLLRHLKIHTREKPFTYSECGKKYSQIVNLASHMKIH",
           refine_extended=True) # On by default, allows unexpectedly long annotations to be refined

################################################################################
# date/time: 2026-01-24 01:10:05
# model: ProteinSequenceAnnotation/PSALM-2
# device: cuda:0
# refine_extended: True
# beam size: 64
# embedding time: 27.08ms
# decoding time: 1.60ms
# top-1 emission filter: 3/24076 families passed
# score filter ≥0.00: 12/12 domains passed
################################################################################
>>> Query: seq1 (336 aa)
--- Pfam domain hits ---
Score   Model     Description              Pfam      Start    Stop
------------------------------------------------------------------
1.000   zf-C2H2   Zinc finger, C2H2 type   PF00096     146     168
1.000   zf-C2H2   Zinc finger, C2H2 type   PF00096     118     140
1.000   zf-C2H2   Zinc finger, C2H2 type   PF00096      90     112
1.000   zf-C2H2   Zinc finger, C2H2 type   PF00096     230     252
1.000   zf-C2H2   Zinc finger, C2H2 type   PF00096      34      56
1.000   zf-C2H2   Zinc finger, C2H2 type   P

{'seq1': [('PF00096',
   6,
   28,
   0.9994053495394224,
   456.2432709377675,
   1.047977089881897,
   19.246155700403886,
   'full'),
  ('PF00096',
   34,
   56,
   0.9996757819903199,
   458.5876471645014,
   1.047977089881897,
   23.359484646674698,
   'full'),
  ('PF00096',
   62,
   84,
   0.9995905115820125,
   491.5901899952758,
   1.047977089881897,
   21.23354710550039,
   'full'),
  ('PF00096',
   90,
   112,
   0.9997338966646419,
   495.6903077054592,
   1.047977089881897,
   27.658395224622478,
   'full'),
  ('PF00096',
   118,
   140,
   0.9997436707206411,
   494.247815676189,
   1.047977089881897,
   24.328155561839992,
   'full'),
  ('PF00096',
   146,
   168,
   0.999768561661198,
   513.8138368538076,
   1.047977089881897,
   24.1980537966868,
   'full'),
  ('PF00096',
   174,
   196,
   0.9995051403167106,
   524.6804258708897,
   1.047977089881897,
   19.352595721950433,
   'full'),
  ('PF00096',
   202,
   224,
   0.9993178644744289,
   492.4447880241136,
   1.0

## Optional parameters

In [ ]:
results = psalm.scan(
    fasta=fasta_path,
    beam_size=64,
    refine_extended=True,
    score_thresh=0.0,
    verbose=False,
)

# Optionally enable FlashAttention if faesm is installed and CUDA is available.
psalm_fa = PSALM(model_name=model_dir, use_fa=True)

## Output format

Each domain entry is a tuple:

```
(pfam, start, stop, cbm_score, bit_score, len_ratio, bias, status)
```

The printed table sorts by CBM `Score`, while `Bit score` corresponds to the PF-LLD bitscore from decoding.

## Loading from Hugging Face

If `model_name` points to a Hugging Face repo ID, PSALM will download the bundle (requires `huggingface_hub`).
Example:

```
psalm = PSALM(model_name="your-org/your-model-repo")
```